In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from datasets import Dataset

# 1. 读取数据
train_df = pd.read_csv("train.csv")
btest_df = pd.read_csv("Btest.csv")



# 2. 划分训练集 / 验证集（比如 9:1）
train_df, val_df = train_test_split(
    train_df,
    test_size=0.1,
    random_state=42,
    stratify=train_df["label"]
)

# 3. 转成 HuggingFace Dataset（方便和 transformers 对接）
train_dataset = Dataset.from_pandas(train_df.reset_index(drop=True))
val_dataset = Dataset.from_pandas(val_df.reset_index(drop=True))
btest_dataset = Dataset.from_pandas(btest_df.reset_index(drop=True))

print(train_dataset[0])

{'id': 1558, 'text': '摸奶节是中国云南双柏县鄂家镇彝族传统文化的庆典就是农历的7月14日、15日与16日这三天，包括外来的游人,如果在街上遇见喜欢的女子，都可以摸一摸女子的胸部。姑娘们表面躲躲闪闪，但决无责怪之意因为这是他们这个地区的百姓延续了1000多年的风俗。小伙子以摸到奶为吉祥，姑娘们以被摸奶为幸运和祝福!', 'label': 1}


In [2]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# 选用 RoFormer 中文 base 模型
model_name = "junnyu/roformer_chinese_base"

# 自动选择对应的 RoFormerTokenizerFast
tokenizer = AutoTokenizer.from_pretrained(model_name)

# 二分类任务，num_labels=2
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)


Some weights of RoFormerForSequenceClassification were not initialized from the model checkpoint at junnyu/roformer_chinese_base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight', 'roformer.encoder.embed_positions.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [3]:
max_length = 256  # 注意：必须与训练时一致（你 BERT 用 128 就保持 128）

def preprocess_function(examples):
    result = tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=max_length
    )
    return result

train_encoded = train_dataset.map(preprocess_function, batched=True)
val_encoded   = val_dataset.map(preprocess_function, batched=True)
btest_encoded = btest_dataset.map(preprocess_function, batched=True)


Parameter 'function'=<function preprocess_function at 0x00000156F3849EE0> of the transform datasets.arrow_dataset.Dataset._map_single couldn't be hashed properly, a random hash was used instead. Make sure your transforms and parameters are serializable with pickle or dill for the dataset fingerprinting and caching to work. If you reuse this transform, the caching mechanism will consider it to be different from the previous calls and recompute everything. This warning is only shown once. Subsequent hashing failures won't be shown.


Map:   0%|          | 0/3305 [00:00<?, ? examples/s]

Map:   0%|          | 0/368 [00:00<?, ? examples/s]

Map:   0%|          | 0/1225 [00:00<?, ? examples/s]

In [4]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds)
    probs = logits[:, 1]
    try:
        auc = roc_auc_score(labels, probs)
    except ValueError:
        auc = float("nan")
    return {"accuracy": acc, "f1": f1, "auc": auc}


In [5]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./roformer_256_bs32",   # RoFormer 的输出目录
    num_train_epochs=4,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,                 # 先用 2e-5，如果有时间可以再开个 3e-5 实验
    weight_decay=0.01,
    logging_dir="./roformer_256_bs32_logs",
    logging_steps=50,
    fp16=True      # 如果报和之前类似的 TypeError 或不支持，就删掉这一行
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_encoded,
    eval_dataset=val_encoded,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)


C:\Users\fall\AppData\Local\Temp\ipykernel_21664\3536238935.py:15: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [6]:
# 训练 RoFormer 模型
trainer.train()

# 在验证集上评估
metrics = trainer.evaluate()
print(metrics)


Step,Training Loss
50,0.466900
100,0.283200
150,0.187900
200,0.135700
250,0.091000
300,0.067400
350,0.064900
400,0.034900


{'eval_loss': 0.18421444296836853, 'eval_accuracy': 0.9456521739130435, 'eval_f1': 0.9431818181818182, 'eval_auc': 0.9900165896433227, 'eval_runtime': 7.9344, 'eval_samples_per_second': 46.38, 'eval_steps_per_second': 0.756, 'epoch': 4.0}


In [7]:
import torch
import numpy as np

predictions = trainer.predict(btest_encoded)

logits = predictions.predictions          # shape: [N, 2]
probs = torch.softmax(torch.tensor(logits), dim=-1).numpy()

prob_fake = probs[:, 1]                   # 正类（label=1=假新闻）概率


submission_b = pd.DataFrame({
    "id": btest_df["id"],
    "prob": prob_fake
})

submission_b.to_csv("roformer_submission_b.csv", index=False, encoding="utf-8")
print("Saved: roformer_submission_b.csv")



Saved: roformer_submission_b.csv
